In [ ]:
%load_ext autoreload
%autoreload 2
# %matplotlib widget
%pdb off

from matplotlib import pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display, HTML
import pyafn
from pyafn import rho, Cd
import random
from scipy.stats import lognorm
from scipy.stats import norm
from scipy.optimize import curve_fit
from emulationHelpers import readEmulationMI, plot_ventilation_model_fit, plot_empirical_model_error_distribution

#close all figures
plt.close('all')
plt.rcParams['figure.dpi'] = 140
im_scaling = .75
plt.rcParams['figure.figsize'] = [6.4 * im_scaling, 4.8 * im_scaling]
plt.rcParams['font.family'] = 'Times New Roman'

home_dir = "./"
display(home_dir)


In [ ]:
flowStatsMI, roomVentilationMI = readEmulationMI(home_dir=home_dir)

y_var = "flux-Norm"
x_var = "p-noInt_optp0-q_model-Norm"

In [ ]:
fig, ax, error_stats = plot_empirical_model_error_distribution(
    flowStatsMI,
    y_var=y_var,
    x_var=x_var,
    bins=50,
    title="Initial Empirical Distribution of Model Error",
)
print(
    f"Empirical error fit: mu={error_stats['mu_hat']:.4f}, "
    f"sigma={error_stats['sigma_hat']:.4f}, n={error_stats['n']}"
)
plt.show()


In [ ]:
from pathlib import Path

from emulationHelpers import (
    fit_bayesian_ventilation_p_subgroups,
    load_bayesian_ventilation_fit_results,
    plot_bayesian_ventilation_p_fit_results,
    plot_bayesian_ventilation_parameter_posteriors,
    save_bayesian_ventilation_fit_results,
    summarize_bayesian_ventilation_fits,
)

a_sigma = 0.5
p_rms_sigma = 0.1
obs_sigma = 0.1

sample_kwargs = {
    "draws": 1000,
    "tune": 1000,
    "chains": 4,
    "cores": 4,
    "progressbar": True,
}

cache_dir = Path("mcmc_cache") / "pressure_scalar_posteriors_alpha_lognormal"
rerun_mcmc = True

if cache_dir.exists() and not rerun_mcmc:
    bayes_fits = load_bayesian_ventilation_fit_results(cache_dir)
    print(f"Loaded cached Bayesian fits from {cache_dir}")
else:
    bayes_fits = fit_bayesian_ventilation_p_subgroups(
        data=flowStatsMI,
        y_var=y_var,
        x_var=x_var,
        a_sigma=a_sigma,
        p_rms_sigma=p_rms_sigma,
        obs_sigma=obs_sigma,
        sample_kwargs=sample_kwargs,
        random_seed=42,
    )
    save_bayesian_ventilation_fit_results(bayes_fits, cache_dir)
    print(f"Saved Bayesian fits to {cache_dir}")


In [ ]:
fig, axs = plot_bayesian_ventilation_p_fit_results(
    data=flowStatsMI,
    fit_results=bayes_fits,
    y_var=y_var,
    x_var=x_var,
    hue="roomType",
    style="slAll",
    model_name="Bayesian Pressure Ventilation Model",
    credible_interval=0.95,
    posterior_draws_for_curves=400,
    show_scatter=True,
    scatter_alpha=0.45,
    band_alpha=0.35,
)

bayes_summary = summarize_bayesian_ventilation_fits(bayes_fits)
display(bayes_summary)
plt.show()


In [ ]:
fig, axs = plot_bayesian_ventilation_p_fit_results(
    data=flowStatsMI,
    fit_results=bayes_fits,
    y_var=y_var,
    x_var=x_var,
    hue="roomType",
    style="slAll",
    credible_interval=0.95,
    posterior_draws_for_curves=400,
    show_scatter=False,
    band_alpha=0.45,
)
plt.show()


In [ ]:
fig, axs = plot_bayesian_ventilation_parameter_posteriors(
    bayes_fits,
    columns=["a", "p_rms", "sigma_obs"],
    kde=True,
)
plt.show()
